<a href="https://colab.research.google.com/github/Timang419/deep-learning-for-mortgage-/blob/main/Script2_standardize_full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import h5py
import os
import json
import pickle
import gc
from sklearn.preprocessing import StandardScaler
import hdf5plugin

paths

In [ ]:
HDF5_DIR    = '/data/math-deep-learning-course/linc6017/dissertation/hdf5/full'
N_SHARDS    = 10
SHARD_PATHS = [os.path.join(HDF5_DIR, f'train_shard_{i}.h5') for i in range(N_SHARDS)]
VAL_H5      = os.path.join(HDF5_DIR, 'val.h5')
TEST_H5     = os.path.join(HDF5_DIR, 'test.h5')
SCALER_PKL  = os.path.join(HDF5_DIR, 'scaler.pkl')
COLS_JSON   = os.path.join(HDF5_DIR, 'feature_cols.json')

RANDOM_SEED = 42
TRANS_CHUNK = 20_000_000

In [ ]:
BINARY_PREFIXES = (
    'state_', 'mod_flag_', 'step_mod_', 'deferral_', 'assist_',
    'FTHB_', 'UNITS_', 'OCC_', 'CHAN_', 'PROP_', 'PURP_',
    'BORR_', 'PROG_', 'VAL_', 'MI_CANCEL_', 'STATE_',
    'VINT_',    # Origination Vintage dummies
)

BINARY_EXACT = {
    'MSA',
    'Prepayment Penalty Mortgage (PPM) Flag',
    'Super Conforming Flag',
    'HARP Indicator',
    'Interest Only (I/O) Indicator',
    'MI_OUT_OF_RANGE',
    'CLTV_MISSING',
    'DTI_MISSING',
    'Delinquency Due to Disaster',
    'Reporting_Period_Int',   # YYYYMM date integer — dropped at training time
}

In [ ]:
def get_continuous_indices(col_names):
    """Return indices of columns that should be standardized (non-binary)."""
    return [
        i for i, col in enumerate(col_names)
        if not any(col.startswith(p) for p in BINARY_PREFIXES)
        and col not in BINARY_EXACT
    ]

In [ ]:
def shuffle_hdf5_full(path, seed):
    """
    Shuffle a train shard fully in-place.
    Identical to sample script2 — safe because each shard is ~49 GB on 256 GB nodes.
    """
    with h5py.File(path, 'a') as f:
        n    = f['X'].shape[0]
        perm = np.random.default_rng(seed).permutation(n)

        X        = f['X'][:]
        y        = f['y'][:]
        loan_ids = f['loan_ids'].asstr()[:]

        f['X'][:]        = X[perm]
        f['y'][:]        = y[perm]
        f['loan_ids'][:] = loan_ids[perm]

    del X, y, loan_ids; gc.collect()
    print(f"  Shuffled (full): {os.path.basename(path)}  ({n:,} rows)")

In [ ]:
def shuffle_hdf5_chunked(path, seed):
    """
    Shuffle HDF5 dataset out-of-core using sweep-and-accumulate.
    Only sequential HDF5 reads; numpy handles fancy indexing in RAM.
    """
    OUT_CHUNK = 20_000_000
    IN_CHUNK  = 40_000_000

    with h5py.File(path, 'r') as f:
        n          = f['X'].shape[0]
        n_features = f['X'].shape[1]

    rng  = np.random.default_rng(seed)
    perm = rng.permutation(n)

    dt_str     = h5py.special_dtype(vlen=str)
    lz4_kwargs = {k: v for k, v in hdf5plugin.LZ4().items() if k != 'chunks'}

    with h5py.File(path, 'r') as f_read, \
         h5py.File(path + '.tmp', 'w') as f_write:

        f_write.create_dataset('X', shape=(n, n_features), dtype=np.float32,
                               chunks=(min(10_000, n), n_features), **lz4_kwargs)
        f_write.create_dataset('y', shape=(n,), dtype=np.int8,
                               chunks=(min(10_000, n),), **lz4_kwargs)
        f_write.create_dataset('loan_ids', shape=(n,), dtype=dt_str,
                               chunks=(min(10_000, n),))

        if 'col_names' in f_read['X'].attrs:
            f_write['X'].attrs['col_names'] = list(f_read['X'].attrs['col_names'])

        for out_start in range(0, n, OUT_CHUNK):
            out_end        = min(out_start + OUT_CHUNK, n)
            target_indices = perm[out_start:out_end]
            sort_order     = np.argsort(target_indices)
            sorted_target  = target_indices[sort_order]
            restore_order  = np.argsort(sort_order)

            chunk_size       = len(target_indices)
            X_chunk_sorted   = np.empty((chunk_size, n_features), dtype=np.float32)
            y_chunk_sorted   = np.empty((chunk_size,), dtype=np.int8)
            ids_chunk_sorted = np.empty((chunk_size,), dtype=object)
            ptr = 0

            for in_start in range(0, n, IN_CHUNK):
                in_end    = min(in_start + IN_CHUNK, n)
                idx_start = np.searchsorted(sorted_target, in_start, side='left')
                idx_end   = np.searchsorted(sorted_target, in_end,   side='left')

                if idx_start < idx_end:
                    local_targets = sorted_target[idx_start:idx_end] - in_start
                    num_found     = len(local_targets)

                    X_in   = f_read['X'][in_start:in_end]
                    y_in   = f_read['y'][in_start:in_end]
                    ids_in = f_read['loan_ids'][in_start:in_end]

                    X_chunk_sorted[ptr:ptr+num_found] = X_in[local_targets]
                    y_chunk_sorted[ptr:ptr+num_found] = y_in[local_targets]

                    raw_bytes = ids_in[local_targets]
                    decoded   = np.array(
                        [x.decode('utf-8') if isinstance(x, bytes) else x for x in raw_bytes],
                        dtype=object)
                    ids_chunk_sorted[ptr:ptr+num_found] = decoded

                    ptr += num_found
                    del X_in, y_in, ids_in, raw_bytes, decoded; gc.collect()

            # Guard against searchsorted off-by-one — uninitialised np.empty
            # garbage would otherwise be silently written to the output file
            assert ptr == chunk_size, \
                f"BUG: ptr={ptr} != chunk_size={chunk_size} at out_start={out_start}"

            f_write['X'][out_start:out_end]        = X_chunk_sorted[restore_order]
            f_write['y'][out_start:out_end]        = y_chunk_sorted[restore_order]
            f_write['loan_ids'][out_start:out_end] = ids_chunk_sorted[restore_order]
            print(f"    progress: {(out_end/n)*100:.1f}%", flush=True)
            del X_chunk_sorted, y_chunk_sorted, ids_chunk_sorted; gc.collect()

    os.replace(path + '.tmp', path)
    del perm; gc.collect()
    print(f"  Shuffled (sweep): {os.path.basename(path)}  ({n:,} rows)")

# **main**

In [ ]:
def main():
    print("=" * 60)
    print("Script 2 (Full): Within-Shard Shuffle + Standardization")
    print("=" * 60)

    # ── Read col_names and feature count from shard_0 ─────────────────────────
    with h5py.File(SHARD_PATHS[0], 'r') as f:
        raw_cols   = list(f['X'].attrs['col_names'])
        col_names  = [c.decode('utf-8') if isinstance(c, bytes) else c for c in raw_cols]
        n_features = f['X'].shape[1]

    print(f"\nFeatures: {n_features}")
    cont_idx    = get_continuous_indices(col_names)
    print(f"Continuous columns to scale : {len(cont_idx)}")
    print(f"Binary columns kept as-is   : {n_features - len(cont_idx)}")
    print(f"Continuous cols: {[col_names[i] for i in cont_idx]}")

    # ── Pass 1: Within-shard shuffle ──────────────────────────────────────────
    print("\n" + "=" * 60)
    print("Pass 1: Within-shard shuffle")
    print("=" * 60)

    for i, path in enumerate(SHARD_PATHS):
        shuffle_hdf5_chunked(path, seed=RANDOM_SEED + i)

    shuffle_hdf5_chunked(VAL_H5,  seed=RANDOM_SEED + N_SHARDS)
    shuffle_hdf5_chunked(TEST_H5, seed=RANDOM_SEED + N_SHARDS + 1)

    # Write col_names to val/test — script1 never sets this attribute because
    # col_names_written is True before any val/test row is first encountered.
    for path in [VAL_H5, TEST_H5]:
        with h5py.File(path, 'r+') as f:
            if 'col_names' not in f['X'].attrs:
                f['X'].attrs['col_names'] = col_names
                print(f"  Written col_names → {os.path.basename(path)}")

    # ── Pass 2: Fit scaler on train shards only ───────────────────────────────
    print("\n" + "=" * 60)
    print("Pass 2: Fitting scaler on train shards (partial_fit)")
    print("=" * 60)

    scaler = StandardScaler()

    for i, path in enumerate(SHARD_PATHS):
        with h5py.File(path, 'r') as f:
            n_rows = f['X'].shape[0]
            for start in range(0, n_rows, TRANS_CHUNK):
                end          = min(start + TRANS_CHUNK, n_rows)
                X_cont_chunk = f['X'][start:end, :].astype(np.float64)[:, cont_idx]
                scaler.partial_fit(X_cont_chunk)
                del X_cont_chunk
            gc.collect()
        print(f"  partial_fit: shard_{i}  ({i + 1}/{N_SHARDS})")

    print("  Scaler fitted.")

    # ── Pass 3: Transform all splits in place ─────────────────────────────────
    print("\n" + "=" * 60)
    print("Pass 3: Applying scaler to all splits")
    print("=" * 60)

    # Train shards — chunked (same approach as val/test)
    for i, path in enumerate(SHARD_PATHS):
        with h5py.File(path, 'r+') as f:
            n_rows = f['X'].shape[0]
            for start in range(0, n_rows, TRANS_CHUNK):
                end   = min(start + TRANS_CHUNK, n_rows)
                chunk = f['X'][start:end, :].astype(np.float64)
                chunk[:, cont_idx] = scaler.transform(chunk[:, cont_idx])
                f['X'][start:end, :] = chunk.astype(np.float32)
            del chunk; gc.collect()
        print(f"  Transformed: train_shard_{i}")

    # Val and test — chunked (too large for full in-memory load)
    for label, path in [('val', VAL_H5), ('test', TEST_H5)]:
        with h5py.File(path, 'r') as f:
            n_rows = f['X'].shape[0]
        print(f"  Transforming {label} ({n_rows:,} rows, chunked) ...", flush=True)
        with h5py.File(path, 'r+') as f:
            for start in range(0, n_rows, TRANS_CHUNK):
                end   = min(start + TRANS_CHUNK, n_rows)
                chunk = f['X'][start:end, :].astype(np.float64)
                chunk[:, cont_idx] = scaler.transform(chunk[:, cont_idx])
                f['X'][start:end, :] = chunk.astype(np.float32)
            del chunk; gc.collect()
        print(f"  Transformed: {label}")

    # ── Save scaler and column list ────────────────────────────────────────────
    with open(SCALER_PKL, 'wb') as f:
        pickle.dump({'scaler': scaler, 'continuous_indices': cont_idx}, f)
    print(f"\nSaved: {SCALER_PKL}")

    with open(COLS_JSON, 'w') as f:
        json.dump(col_names, f, indent=2)
    print(f"Saved: {COLS_JSON}")

    # ── Summary ────────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("Done.")
    total_train = 0
    for i, path in enumerate(SHARD_PATHS):
        with h5py.File(path, 'r') as f:
            nr = f['X'].shape[0]
            total_train += nr
            print(f"  train_shard_{i}: {nr:,} rows")
    for label, path in [('val', VAL_H5), ('test', TEST_H5)]:
        with h5py.File(path, 'r') as f:
            print(f"  {label}:          {f['X'].shape[0]:,} rows")
    print(f"  ─────────────────────────────────")
    print(f"  Total train : {total_train:,} rows")
    print(f"  Features    : {n_features}")
    print(f"  Scaled cols : {len(cont_idx)}")
    print(f"  Scaler      : {SCALER_PKL}")
    print(f"  Cols JSON   : {COLS_JSON}")
    print("=" * 60)

    # ── Validation: chunk layout + NaN scan ───────────────────────────────────
    print("\n" + "=" * 60)
    print("Validation checks")
    print("=" * 60)

    all_ok = True

    for label, path in (
        [(f'train_shard_{i}', p) for i, p in enumerate(SHARD_PATHS)]
        + [('val', VAL_H5), ('test', TEST_H5)]
    ):
        with h5py.File(path, 'r') as f:
            # ── 1. Chunk shape: first dim must be >> second dim (row-oriented) ──
            chunk_shape = f['X'].chunks          # e.g. (10000, 80)
            n_rows_file = f['X'].shape[0]
            n_cols_file = f['X'].shape[1]

            if chunk_shape is None:
                print(f"  [WARN] {label}: X has no chunking at all")
                all_ok = False
            elif chunk_shape[0] < chunk_shape[1]:
                # Column-oriented: chunk rows < chunk cols  e.g. (1, 80) or (2, 80)
                print(f"  [FAIL] {label}: chunk shape {chunk_shape} is COLUMN-oriented "
                      f"(rows {chunk_shape[0]} < cols {chunk_shape[1]}) — BAD")
                all_ok = False
            else:
                print(f"  [OK]   {label}: chunk shape {chunk_shape}  "
                      f"({n_rows_file:,} rows × {n_cols_file} cols)")

            # ── 2. NaN scan per column (chunked to stay within RAM) ────────────
            # Dict accumulates counts across chunks so each column is reported
            # once with its true total, not once per chunk.
            nan_totals = {}   # col_idx → cumulative NaN count
            for start in range(0, n_rows_file, TRANS_CHUNK):
                end        = min(start + TRANS_CHUNK, n_rows_file)
                X_chunk    = f['X'][start:end, :].astype(np.float32)
                nan_counts = np.isnan(X_chunk).sum(axis=0)   # shape: (n_cols,)
                for col_idx, cnt in enumerate(nan_counts):
                    if cnt > 0:
                        nan_totals[col_idx] = nan_totals.get(col_idx, 0) + int(cnt)
                del X_chunk; gc.collect()

            if nan_totals:
                print(f"  [FAIL] {label}: NaN found in {len(nan_totals)} column(s):")
                for col_idx, cnt in sorted(nan_totals.items()):
                    print(f"           col {col_idx:>3d}  '{col_names[col_idx]}'  →  {cnt:,} NaN rows")
                all_ok = False
            else:
                print(f"  [OK]   {label}: no NaN values found")

    print("=" * 60)
    if all_ok:
        print("ALL CHECKS PASSED — safe to run Script 3.")
    else:
        print("ONE OR MORE CHECKS FAILED — fix before running Script 3.")
    print("=" * 60)


if __name__ == '__main__':
    main()